# DDI Preprocessing Pipeline

Notebook version of `ddi_preprocessing_pipeline.py`. This notebook keeps the same preprocessing logic and is organized into reusable sections.

## 1. Imports

In [14]:
import re
import numpy as np
import pandas as pd

## 2. Text Cleaning and Drug Name Normalization

In [15]:
"""
DDI preprocessing pipeline.
- Preserves raw values.
- Normalizes drug names.
- Creates deterministic unordered pair IDs for grouping/splitting.
- Preserves directional drug1 -> drug2 semantics.
- Extracts only conservative text-derived features.
- Never invents severity.
- Keeps an audit trail for duplicates/conflicts.
"""

def clean_text(value):
    if pd.isna(value): return None
    value = str(value).replace("\u00a0", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value or None

def normalize_drug_name(value):
    value = clean_text(value)
    if value is None: return None
    value = value.lower()
    value = re.sub(r"[‐‑‒–—−]", "-", value)
    value = re.sub(r"\s*([,/+])\s*", r"\1", value)
    value = re.sub(r"\s*-\s*", "-", value)
    value = re.sub(r"\s+", " ", value).strip(" .")
    return value



## 3. Label and Pair Utilities

In [16]:
def normalize_label(value):
    if pd.isna(value) or not str(value).strip(): return pd.NA
    x = re.sub(r"\s+", " ", str(value).strip().lower())
    return {
        "low": "low", "low risk": "low",
        "moderate": "moderate", "moderate risk": "moderate",
        "high": "high", "high risk": "high",
    }.get(x, pd.NA)

def make_pair_id(a, b):
    if not a or not b: return None
    return "|".join(sorted([a, b]))

## 4. Drug Entity Resolution with a Reference Master

Fuzzy matching is only a candidate-generation step. Matches should be medically/independently validated before being treated as canonical identities.

In [17]:
def resolve_against_reference(query_names, reference_names,
                              high_threshold=95, possible_threshold=85):
    """
    Hierarchical entity resolution against an external drug master:
      1) exact normalized match
      2) RapidFuzz candidate scoring
      3) ambiguous/unmatched status
    Returns a DataFrame that must be manually/medically validated before
    fuzzy matches are promoted to canonical identities.
    """
    try:
        from rapidfuzz import process, fuzz
    except ImportError as exc:
        raise ImportError("Install RapidFuzz for fuzzy entity resolution.") from exc

    refs = pd.Series(reference_names).dropna().map(normalize_drug_name).drop_duplicates().tolist()
    refset = set(refs)
    rows = []

    for raw_name in query_names:
        q = normalize_drug_name(raw_name)
        if not q:
            rows.append({"query": raw_name, "normalized": q, "match": None,
                         "score": None, "status": "unmatched"})
            continue

        if q in refset:
            rows.append({"query": raw_name, "normalized": q, "match": q,
                         "score": 100.0, "status": "exact_match"})
            continue

        candidates = process.extract(q, refs, scorer=fuzz.ratio, limit=3)
        if not candidates:
            rows.append({"query": raw_name, "normalized": q, "match": None,
                         "score": None, "status": "unmatched"})
            continue

        best = candidates[0]
        second = candidates[1] if len(candidates) > 1 else None
        best_name, best_score = best[0], float(best[1])
        second_score = float(second[1]) if second else -1.0

        if best_score >= high_threshold and (best_score - second_score >= 3):
            status = "high_confidence_match"
            match = best_name
        elif best_score >= possible_threshold:
            status = "ambiguous_match" if second and (best_score - second_score < 3) else "possible_match"
            match = best_name
        else:
            status = "unmatched"
            match = None

        rows.append({
            "query": raw_name,
            "normalized": q,
            "match": match,
            "score": best_score,
            "status": status
        })

    return pd.DataFrame(rows).drop_duplicates("normalized")


def attach_reference_master(processed, reference_master,
                            query_col="normalized_name",
                            canonical_col="canonical_drug",
                            active_ingredient_col=None,
                            class_col=None,
                            strength_col=None,
                            form_col=None):
    """
    Join an independently curated pharmaceutical master AFTER entity
    resolution. Do not use fuzzy output as truth without validation.
    """
    master = reference_master.copy()
    master[query_col] = master[query_col].map(normalize_drug_name)

    cols = [query_col, canonical_col]
    for c in [active_ingredient_col, class_col, strength_col, form_col]:
        if c and c in master.columns:
            cols.append(c)

    m = master[cols].drop_duplicates(query_col).copy()
    m = m.rename(columns={canonical_col: "_master_canonical"})

    result = processed.copy()
    result = result.merge(
        m, left_on="drug1_normalized", right_on=query_col, how="left"
    ).drop(columns=[query_col], errors="ignore")
    result = result.rename(columns={"_master_canonical": "drug1_canonical_master"})

    result = result.merge(
        m, left_on="drug2_normalized", right_on=query_col, how="left",
        suffixes=("", "_drug2")
    ).drop(columns=[query_col], errors="ignore")
    result = result.rename(columns={"_master_canonical": "drug2_canonical_master"})
    return result



## 5. Interaction Feature Extraction

In [18]:
EFFECT_PATTERNS = {
    "increase": r"\b(?:increase|increased|increases|increasing|higher|elevat\w*|enhanc\w*)\b",
    "decrease": r"\b(?:decrease|decreased|decreases|decreasing|lower|reduc\w*|diminish\w*)\b",
    "no_change": r"\b(?:no change|unchanged|does not change|without changing)\b",
}

MECHANISM_PATTERNS = [
    ("metabolism", r"\bmetabolism|metabol\w*\b"),
    ("concentration", r"\bserum concentration|serum level|concentration\b"),
    ("absorption", r"\babsorption|absorbed\b"),
    ("excretion", r"\bexcretion|excret\w*\b"),
    ("toxicity", r"\btoxic\w*\b"),
    ("therapeutic_effect", r"\btherapeutic (?:efficacy|effect|effects)|efficacy\b"),
    ("activity", r"\bactivities\b|\bactivity\b"),
    ("adverse_effect", r"\badverse effects?\b"),
]

CATEGORY_PATTERNS = [
    ("adverse_effect", r"\badverse effects?\b"),
    ("photosensitizing", r"\bphotosensitiz\w*"),
    ("cardiotoxic", r"\bcardiotoxic\w*"),
    ("hypotensive", r"\bhypotensive\b"),
    ("qtc_prolonging", r"\bqtc-prolonging\b"),
    ("cns_depressant", r"\bcentral nervous system depressant|cns depressant\b"),
    ("anticoagulant", r"\banticoagulant\b"),
    ("hypoglycemic", r"\bhypoglycemic\b"),
    ("antihypertensive", r"\bantihypertensive\b"),
    ("bradycardic", r"\bbradycardic\b"),
    ("hypokalemic", r"\bhypokalemic\b"),
    ("sedative", r"\bsedative\b"),
    ("neuroexcitatory", r"\bneuroexcitatory\b"),
    ("serotonergic", r"\bserotonergic\b"),
]

def extract_effect_direction(text):
    if not text: return "unknown"
    text = text.lower()
    inc = bool(re.search(EFFECT_PATTERNS["increase"], text))
    dec = bool(re.search(EFFECT_PATTERNS["decrease"], text))
    no_change = bool(re.search(EFFECT_PATTERNS["no_change"], text))
    if no_change and not (inc or dec): return "no_change"
    if inc and not dec: return "increase"
    if dec and not inc: return "decrease"
    return "unknown"

def extract_multi(text, patterns):
    if not text: return "unknown"
    text = text.lower()
    hits = [name for name, pattern in patterns if re.search(pattern, text)]
    return "|".join(hits) if hits else "unknown"



## 6. Main Preprocessing Function

In [19]:
def preprocess_ddi(df, severity_col=None, reference_drugs=None):
    required = {"Drug 1", "Drug 2", "Interaction Description"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    out = pd.DataFrame(index=df.index)
    out["drug1_raw"] = df["Drug 1"].map(clean_text)
    out["drug2_raw"] = df["Drug 2"].map(clean_text)
    out["interaction_description_raw"] = df["Interaction Description"].map(clean_text)

    # Invalid rows are tracked in the audit, not kept in the main processed dataset.
    invalid_record = (
        out["drug1_raw"].isna() | out["drug2_raw"].isna() |
        out["interaction_description_raw"].isna()
    )

    out["drug1_normalized"] = out["drug1_raw"].map(normalize_drug_name)
    out["drug2_normalized"] = out["drug2_raw"].map(normalize_drug_name)
    out["drug1_canonical"] = out["drug1_normalized"]
    out["drug2_canonical"] = out["drug2_normalized"]
    out["drug_pair_id"] = [
        make_pair_id(a, b) for a, b in
        zip(out["drug1_canonical"], out["drug2_canonical"])
    ]
    out["canonical_drug1"] = [
        sorted([a, b])[0] if a and b else None
        for a, b in zip(out["drug1_canonical"], out["drug2_canonical"])
    ]
    out["canonical_drug2"] = [
        sorted([a, b])[1] if a and b else None
        for a, b in zip(out["drug1_canonical"], out["drug2_canonical"])
    ]

    out["interaction_description_clean"] = out["interaction_description_raw"].map(clean_text)
    out["effect_direction"] = out["interaction_description_clean"].map(extract_effect_direction)
    out["interaction_mechanism"] = out["interaction_description_clean"].map(
        lambda x: extract_multi(x, MECHANISM_PATTERNS)
    )
    out["interaction_category"] = out["interaction_description_clean"].map(
        lambda x: extract_multi(x, CATEGORY_PATTERNS)
    )

    # Pharmaceutical metadata is intentionally not added when it is unavailable.
    # It must come from a real reference/master dataset before being included.

    if severity_col and severity_col in df.columns:
        severity = df[severity_col].map(normalize_label)
        severity_invalid = df[severity_col].notna() & severity.isna()
    else:
        severity = pd.Series(pd.NA, index=df.index, dtype="object")
        severity_invalid = pd.Series(False, index=df.index)

    exact_key = (
        out["drug1_normalized"].fillna("") + "||" +
        out["drug2_normalized"].fillna("") + "||" +
        out["interaction_description_clean"].fillna("")
    )
    exact_duplicate = exact_key.duplicated(keep="first")

    pair_desc_count = out.groupby("drug_pair_id")["interaction_description_clean"].transform("nunique")
    out["pair_has_multiple_descriptions"] = pair_desc_count.gt(1)

    directed_pairs = set(zip(out["drug1_normalized"], out["drug2_normalized"]))
    out["pair_is_directionally_reversed_elsewhere"] = [
        (b, a) in directed_pairs if a and b else False
        for a, b in zip(out["drug1_normalized"], out["drug2_normalized"])
    ]

    if reference_drugs is None:
        out["entity_match_status_drug1"] = np.where(
            out["drug1_normalized"].notna(), "exact_normalized_self", "unmatched"
        )
        out["entity_match_status_drug2"] = np.where(
            out["drug2_normalized"].notna(), "exact_normalized_self", "unmatched"
        )
    else:
        refs = pd.Series(reference_drugs).dropna().map(normalize_drug_name).drop_duplicates()
        refset = set(refs)
        out["entity_match_status_drug1"] = np.where(
            out["drug1_normalized"].isin(refset), "exact_match", "unmatched"
        )
        out["entity_match_status_drug2"] = np.where(
            out["drug2_normalized"].isin(refset), "exact_match", "unmatched"
        )

    # Audit keeps all diagnostic information, including fields that are not part
    # of the main modeling/Ollama dataset.
    audit = out.copy()
    audit["invalid_record"] = invalid_record
    audit["exact_normalized_duplicate"] = exact_duplicate
    audit["severity_label"] = severity
    audit["severity_label_invalid"] = severity_invalid

    # Main processed dataset: only useful, currently populated processing fields.
    processed = out.loc[~invalid_record & ~exact_duplicate].copy()

    return processed, audit


## 7. Run Preprocessing and Save Outputs

This cell runs the pipeline and writes the main processed dataset plus the separate audit file.
The main `ddi_processed.csv` is intentionally limited to the current 18-column schema.


In [20]:
INPUT_PATH = r"C:\Users\Ahmed\Desktop\Learn_Python\Project_DEPI\db_drug_interactions.csv"
OUTPUT_DIR =  r"C:\Users\Ahmed\Desktop\Learn_Python\Project_DEPI\ddi_preprocessing"

EXPECTED_PROCESSED_COLUMNS = [
    "drug1_raw", "drug2_raw", "interaction_description_raw",
    "drug1_normalized", "drug2_normalized",
    "drug1_canonical", "drug2_canonical", "drug_pair_id",
    "canonical_drug1", "canonical_drug2",
    "interaction_description_clean", "effect_direction",
    "interaction_mechanism", "interaction_category",
    "pair_has_multiple_descriptions",
    "pair_is_directionally_reversed_elsewhere",
    "entity_match_status_drug1", "entity_match_status_drug2",
]

df_raw = pd.read_csv(INPUT_PATH)
processed_df, audit_df = preprocess_ddi(df_raw)

# Enforce the exact schema used by the cleaned ddi_processed.csv.
processed_df = processed_df[EXPECTED_PROCESSED_COLUMNS].copy()

processed_path = f"{OUTPUT_DIR}/ddi_processed.csv"
audit_path = f"{OUTPUT_DIR}/ddi_processing_audit.csv"
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)
processed_df.to_csv(processed_path, index=False)
audit_df.to_csv(audit_path, index=False)

assert list(processed_df.columns) == EXPECTED_PROCESSED_COLUMNS

print("Raw shape:", df_raw.shape)
print("Processed shape:", processed_df.shape)
print("Audit shape:", audit_df.shape)
print("Processed schema validated:", list(processed_df.columns) == EXPECTED_PROCESSED_COLUMNS)
processed_df.head()


Raw shape: (191541, 3)
Processed shape: (191541, 18)
Audit shape: (191541, 22)
Processed schema validated: True


,drug1_raw,drug2_raw,interaction_description_raw,drug1_normalized,drug2_normalized,drug1_canonical,drug2_canonical,drug_pair_id,canonical_drug1,canonical_drug2,interaction_description_clean,effect_direction,interaction_mechanism,interaction_category,pair_has_multiple_descriptions,pair_is_directionally_reversed_elsewhere,entity_match_status_drug1,entity_match_status_drug2
0,Trioxsalen,Verteporfin,Trioxsalen may increase the photosensitizing a...,trioxsalen,verteporfin,trioxsalen,verteporfin,trioxsalen|verteporfin,trioxsalen,verteporfin,Trioxsalen may increase the photosensitizing a...,increase,activity,photosensitizing,False,False,exact_normalized_self,exact_normalized_self
1,Aminolevulinic acid,Verteporfin,Aminolevulinic acid may increase the photosens...,aminolevulinic acid,verteporfin,aminolevulinic acid,verteporfin,aminolevulinic acid|verteporfin,aminolevulinic acid,verteporfin,Aminolevulinic acid may increase the photosens...,increase,activity,photosensitizing,False,False,exact_normalized_self,exact_normalized_self
2,Titanium dioxide,Verteporfin,Titanium dioxide may increase the photosensiti...,titanium dioxide,verteporfin,titanium dioxide,verteporfin,titanium dioxide|verteporfin,titanium dioxide,verteporfin,Titanium dioxide may increase the photosensiti...,increase,activity,photosensitizing,False,False,exact_normalized_self,exact_normalized_self
3,Tiaprofenic acid,Verteporfin,Tiaprofenic acid may increase the photosensiti...,tiaprofenic acid,verteporfin,tiaprofenic acid,verteporfin,tiaprofenic acid|verteporfin,tiaprofenic acid,verteporfin,Tiaprofenic acid may increase the photosensiti...,increase,activity,photosensitizing,False,False,exact_normalized_self,exact_normalized_self
4,Cyamemazine,Verteporfin,Cyamemazine may increase the photosensitizing ...,cyamemazine,verteporfin,cyamemazine,verteporfin,cyamemazine|verteporfin,cyamemazine,verteporfin,Cyamemazine may increase the photosensitizing ...,increase,activity,photosensitizing,False,False,exact_normalized_self,exact_normalized_self
